In [1]:
# ============================================================
# LangGraph로 15-message-trimming-and-summarization의 대화 이력 관리 학습 재구성
# ============================================================
#
# 이 코드는 다음 3가지를 한 번에 학습하기 위한 예제입니다.
#
# 1. 기본 대화 이력 관리
#    - 같은 thread_id로 여러 번 호출하면 이전 대화를 기억하는 것처럼 동작
#
# 2. 대화 이력 trimming
#    - 전체 대화는 checkpoint에 저장하지만,
#      모델에게 전달할 때는 최근 메시지만 잘라서 전달
#
# 3. 대화 이력 summarization
#    - 대화가 길어지면 오래된 메시지를 요약하고,
#      원문 메시지는 삭제해서 context를 압축
#
# ============================================================

In [2]:
# ------------------------------------------------------------
# 0. 패키지 import
# ------------------------------------------------------------

# 필요 시 설치
# !pip install -U langgraph langchain-openai langchain-core

from typing import Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    RemoveMessage,
)
from langchain_core.messages.utils import (
    trim_messages,
    count_tokens_approximately,
)

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
# ------------------------------------------------------------
# 1. 모델 설정
# ------------------------------------------------------------
#
# ChatOpenAI를 사용하지만, 실제 OpenAI API가 아니라
# LM Studio의 OpenAI-compatible API 서버로 요청을 보냅니다.
#
# base_url은 사용자의 LM Studio 서버 주소에 맞게 유지합니다.

model = ChatOpenAI(
    model="unsloth/gemma-4-e2b-it",
    base_url="http://...:.../v1",
    api_key="lm-studio",
    temperature=0,
)

In [4]:
# ------------------------------------------------------------
# 2. 공통 system message
# ------------------------------------------------------------
#
# 매번 모델을 호출할 때 대화 이력 앞에 붙일 system message입니다.
# LangGraph의 state["messages"]에는 보통 사용자/AI 대화만 누적시키고,
# system message는 모델 호출 시점에 따로 붙이는 방식이 깔끔합니다.

SYSTEM_MESSAGE = SystemMessage(
    content="당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."
)

In [5]:
# ============================================================
# PART 1. 기본 대화 이력 관리
# ============================================================
#
# 목표:
# - 같은 thread_id로 호출하면 이전 대화가 유지되는지 확인합니다.
#
# 핵심:
# - MessagesState는 messages 필드를 가진 state입니다.
# - graph.compile(checkpointer=memory)를 하면 상태가 저장됩니다.
# - config의 thread_id가 같으면 같은 대화 상태를 이어갑니다.


def call_model_basic(state: MessagesState):
    """
    기본 모델 호출 노드입니다.

    state["messages"]에는 지금까지의 대화 이력이 들어 있습니다.
    여기에 system message를 앞에 붙여 모델에게 전달합니다.

    return {"messages": [response]}를 하면
    LangGraph가 기존 messages 뒤에 response를 누적합니다.
    """

    response = model.invoke(
        [SYSTEM_MESSAGE] + state["messages"]
    )

    return {
        "messages": [response]
    }


# 기본 그래프 생성
basic_builder = StateGraph(MessagesState)

basic_builder.add_node("call_model", call_model_basic)
basic_builder.add_edge(START, "call_model")
basic_builder.add_edge("call_model", END)

In [6]:
# InMemorySaver는 메모리에 checkpoint를 저장합니다.
# 학습/실험용으로 적합합니다.
basic_memory = InMemorySaver()

basic_graph = basic_builder.compile(checkpointer=basic_memory)

In [7]:
# ============================================================
# PART 2. 대화 이력 trimming
# ============================================================
#
# 목표:
# - 전체 대화는 checkpoint에 저장하되,
# - 모델에게 전달할 때는 최근 메시지만 전달합니다.
#
# 15-message-trimming-and-summarization의 trim_messages 실험과 대응됩니다.
#
# 주의:
# - trimming은 저장된 state 자체를 삭제하는 것이 아닙니다.
# - 모델 호출 직전에 사용할 메시지만 줄이는 것입니다.


def call_model_with_trimming(state: MessagesState):
    """
    trimming이 적용된 모델 호출 노드입니다.

    state["messages"] 전체를 모델에 전달하지 않고,
    trim_messages로 최근 메시지만 남긴 뒤 전달합니다.
    """

    # 최근 대화만 남깁니다.
    # max_tokens는 학습용으로 작게 잡아도 되고,
    # 실제 사용에서는 모델 context window에 맞춰 조절합니다.
    trimmed_messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=512,
        start_on="human",
    )

    response = model.invoke(
        [SYSTEM_MESSAGE] + trimmed_messages
    )

    return {
        "messages": [response]
    }

In [8]:
# trimming 그래프 생성
trim_builder = StateGraph(MessagesState)

trim_builder.add_node("call_model", call_model_with_trimming)
trim_builder.add_edge(START, "call_model")
trim_builder.add_edge("call_model", END)

trim_memory = InMemorySaver()

trim_graph = trim_builder.compile(checkpointer=trim_memory)

In [9]:
# ============================================================
# PART 3. 대화 이력 summarization
# ============================================================
#
# 목표:
# - 대화가 길어지면 이전 대화를 요약합니다.
# - 오래된 메시지는 RemoveMessage로 삭제합니다.
# - 요약문은 state["summary"]에 저장합니다.
#
# 15-message-trimming-and-summarization의 summarize_messages 실험과 대응됩니다.
#
# 핵심 차이:
# - 기존 코드는 chat_history.clear() 후 요약 메시지만 넣었습니다.
# - LangGraph에서는 state에 summary 필드를 따로 두고,
#   오래된 messages를 RemoveMessage로 삭제합니다.


class SummaryState(MessagesState):
    """
    MessagesState에 summary 필드를 추가한 상태입니다.

    messages:
        최근 대화 메시지 목록

    summary:
        오래된 대화를 압축한 요약문
    """

    summary: str

In [10]:
def call_model_with_summary(state: SummaryState):
    """
    summary를 참고해서 모델을 호출하는 노드입니다.

    summary가 있으면 system message에 요약 정보를 추가합니다.
    이렇게 하면 모델은 오래된 원문 대화가 없어도
    요약된 맥락을 참고할 수 있습니다.
    """

    summary = state.get("summary", "")

    system_content = "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."

    if summary:
        system_content += f"""

이전 대화 요약:
{summary}

위 요약과 최근 대화 내용을 함께 참고해서 답변하십시오.
"""

    system_message = SystemMessage(content=system_content)

    response = model.invoke(
        [system_message] + state["messages"]
    )

    return {
        "messages": [response]
    }

In [11]:
def should_summarize(state: SummaryState) -> Literal["summarize", "__end__"]:
    """
    대화를 요약할지 결정하는 조건 함수입니다.

    messages가 일정 개수를 넘으면 summarize 노드로 이동합니다.
    그렇지 않으면 종료합니다.
    """

    messages = state["messages"]

    # 학습용 기준입니다.
    # 사용자 메시지 + AI 메시지가 계속 쌓여 6개를 넘으면 요약합니다.
    if len(messages) > 6:
        return "summarize"

    return END

In [12]:
def summarize_conversation(state: SummaryState):
    """
    오래된 대화를 요약하는 노드입니다.

    동작 과정:
    1. 기존 summary가 있으면 가져옵니다.
    2. 현재 messages를 기반으로 요약을 생성합니다.
    3. 오래된 메시지를 RemoveMessage로 삭제합니다.
    4. 새 summary를 state에 저장합니다.
    5. 최근 메시지 일부만 남깁니다.
    """

    existing_summary = state.get("summary", "")

    if existing_summary:
        summary_instruction = f"""
기존 대화 요약은 다음과 같습니다.

{existing_summary}

위 요약을 유지하면서, 최근 대화 내용을 반영해 갱신된 요약을 작성하세요.
사용자의 목표, 질문, AI가 제공한 재정 조언을 가능한 한 보존하세요.
"""
    else:
        summary_instruction = """
지금까지의 대화를 요약하세요.
사용자의 목표, 질문, AI가 제공한 재정 조언을 가능한 한 보존하세요.
"""

    # 요약 생성용 메시지입니다.
    # 기존 대화 뒤에 요약 요청을 추가해서 모델에게 전달합니다.
    summary_messages = state["messages"] + [
        HumanMessage(content=summary_instruction)
    ]

    summary_response = model.invoke(summary_messages)

    # 최근 2개 메시지만 남기고, 그 이전 메시지는 삭제합니다.
    # 보통 최근 사용자 질문 + 최근 AI 답변 정도를 남기는 용도입니다.
    messages_to_delete = [
        RemoveMessage(id=message.id)
        for message in state["messages"][:-2]
        if message.id is not None
    ]

    return {
        "summary": summary_response.content,
        "messages": messages_to_delete,
    }

In [13]:
# summarization 그래프 생성
summary_builder = StateGraph(SummaryState)

summary_builder.add_node("call_model", call_model_with_summary)
summary_builder.add_node("summarize", summarize_conversation)

summary_builder.add_edge(START, "call_model")

# call_model 실행 후, 메시지가 많으면 summarize로 이동합니다.
summary_builder.add_conditional_edges(
    "call_model",
    should_summarize,
    {
        "summarize": "summarize",
        END: END,
    },
)

summary_builder.add_edge("summarize", END)

summary_memory = InMemorySaver()

summary_graph = summary_builder.compile(checkpointer=summary_memory)

In [14]:
# ============================================================
# 디버깅용 함수
# ============================================================

def print_last_answer(result):
    """
    graph.invoke 결과에서 마지막 AI 응답만 출력합니다.
    """

    print(result["messages"][-1].content)


def print_graph_state(graph, config):
    """
    현재 thread_id에 저장된 state를 확인합니다.

    이 함수로 다음을 확인할 수 있습니다.
    - 현재 messages에 무엇이 남아 있는가?
    - summary가 생성되었는가?
    """

    state = graph.get_state(config)

    print("\n===== 현재 summary =====")
    print(state.values.get("summary", ""))

    print("\n===== 현재 messages =====")
    for message in state.values["messages"]:
        print(f"{message.type}: {message.content}")
        print("-" * 50)

In [15]:
# ============================================================
# 실행 예시 1. 기본 대화 이력 관리
# ============================================================

print("\n\n==============================")
print("1. 기본 대화 이력 관리")
print("==============================")

basic_config = {
    "configurable": {
        # LangGraph에서는 thread_id가 실제 대화 상태 저장 key입니다.
        # 같은 thread_id를 사용하면 이전 대화가 이어집니다.
        "thread_id": "basic_finance_session"
    }
}

result = basic_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저축을 늘리기 위해 무엇을 할 수 있나요?")
        ]
    },
    basic_config,
)

print_last_answer(result)



1. 기본 대화 이력 관리
저축을 늘리는 것은 재정적 목표를 달성하고 미래를 안정적으로 만드는 데 매우 중요합니다. 저축액을 늘리기 위한 구체적이고 실용적인 방법들을 단계별로 자세히 안내해 드리겠습니다.

---

## 1단계: 현재 재정 상태 파악하기 (기반 다지기)

저축을 늘리기 전에, **돈이 어디로 가고 있는지** 정확히 아는 것이 가장 중요합니다.

1. **수입과 지출 기록:** 최소 한 달 동안 모든 수입(월급, 부수입 등)과 모든 지출(고정비, 변동비 포함)을 꼼꼼하게 기록하세요. 가계부 앱이나 엑셀 시트를 활용하면 편리합니다.
2. **예산 설정:** 기록을 바탕으로 '필수 지출', '선택적 지출'을 구분하고, 저축 목표액을 정하여 예산을 세우세요. (예: 수입의 50%는 고정 지출, 30%는 변동 지출, **20% 이상은 저축/투자**).
3. **재정 목표 설정:** '왜 저축하는가?'에 대한 명확한 목표(예: 1년 안에 비상금 300만 원 모으기, 5년 뒤 주택 계약금 마련 등)를 설정하면 동기 부여가 됩니다.

## 2단계: 지출 통제 및 절약 (돈을 아끼는 단계)

저축액을 늘리는 가장 직접적인 방법은 **지출을 줄이는 것**입니다.

* **고정 지출 점검:** 구독 서비스(OTT, 앱 등), 보험료, 통신비 등을 정기적으로 검토하여 불필요한 지출을 해지하거나 요금제를 조정하세요.
* **변동 지출 관리 (생활 습관 개선):**
    * **식비 절약:** 외식 횟수를 줄이고 도시락을 싸거나 식단을 계획하여 식비를 절감합니다.
    * **충동구매 방지:** 물건을 사기 전에 '정말 필요한가?', '이것이 저축 목표에 도움이 되는가?'를 자문하고, 구매 전 24시간 규칙 등을 적용해 보세요.
    * **구독 서비스 정리:** 사용하지 않는 멤버십이나 서비스를 과감하게 해지합니다.
* **'선 저축, 후 소비' 원칙:** 월급이 들어오면 가장 먼저 저축할 금액을 별도의 계좌로 이체하고, 남은 돈으로 생활하는 습관을 들이세요

In [16]:
result = basic_graph.invoke(
    {
        "messages": [
            HumanMessage(content="내가 방금 뭐라고 했나요?")
        ]
    },
    basic_config,
)

print_last_answer(result)

방금 고객님께서는 **"저축을 늘리기 위해 무엇을 할 수 있나요?"** 라고 질문하셨습니다.

저는 그 질문에 대해 저축을 늘리기 위한 구체적인 4단계 전략(재정 상태 파악, 지출 통제, 수입 증대, 자동화 시스템 구축)과 실천 체크리스트를 안내해 드렸습니다.


In [17]:
print_graph_state(basic_graph, basic_config)


===== 현재 summary =====


===== 현재 messages =====
human: 저축을 늘리기 위해 무엇을 할 수 있나요?
--------------------------------------------------
ai: 저축을 늘리는 것은 재정적 목표를 달성하고 미래를 안정적으로 만드는 데 매우 중요합니다. 저축액을 늘리기 위한 구체적이고 실용적인 방법들을 단계별로 자세히 안내해 드리겠습니다.

---

## 1단계: 현재 재정 상태 파악하기 (기반 다지기)

저축을 늘리기 전에, **돈이 어디로 가고 있는지** 정확히 아는 것이 가장 중요합니다.

1. **수입과 지출 기록:** 최소 한 달 동안 모든 수입(월급, 부수입 등)과 모든 지출(고정비, 변동비 포함)을 꼼꼼하게 기록하세요. 가계부 앱이나 엑셀 시트를 활용하면 편리합니다.
2. **예산 설정:** 기록을 바탕으로 '필수 지출', '선택적 지출'을 구분하고, 저축 목표액을 정하여 예산을 세우세요. (예: 수입의 50%는 고정 지출, 30%는 변동 지출, **20% 이상은 저축/투자**).
3. **재정 목표 설정:** '왜 저축하는가?'에 대한 명확한 목표(예: 1년 안에 비상금 300만 원 모으기, 5년 뒤 주택 계약금 마련 등)를 설정하면 동기 부여가 됩니다.

## 2단계: 지출 통제 및 절약 (돈을 아끼는 단계)

저축액을 늘리는 가장 직접적인 방법은 **지출을 줄이는 것**입니다.

* **고정 지출 점검:** 구독 서비스(OTT, 앱 등), 보험료, 통신비 등을 정기적으로 검토하여 불필요한 지출을 해지하거나 요금제를 조정하세요.
* **변동 지출 관리 (생활 습관 개선):**
    * **식비 절약:** 외식 횟수를 줄이고 도시락을 싸거나 식단을 계획하여 식비를 절감합니다.
    * **충동구매 방지:** 물건을 사기 전에 '정말 필요한가?', '이것이 저축 목표에 도움이 되는가?'를 자문하고, 구매 전 24시간 규칙 등을 적용해 보세요.
    * **구독 서비

In [18]:
# ============================================================
# 실행 예시 2. trimming 적용
# ============================================================

print("\n\n==============================")
print("2. trimming 적용")
print("==============================")

trim_config = {
    "configurable": {
        "thread_id": "trim_finance_session"
    }
}



2. trimming 적용


In [19]:
result = trim_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저축을 늘리기 위해 무엇을 할 수 있나요?")
        ]
    },
    trim_config,
)

print_last_answer(result)

저축을 늘리는 것은 재정적 목표를 달성하는 데 매우 중요합니다. 단순히 '돈을 아껴야지'라고 생각하기보다는, **체계적인 계획과 습관의 변화**가 필요합니다.

제가 금융 상담사로서, 저축액을 효과적으로 늘릴 수 있는 구체적이고 실질적인 방법들을 단계별로 제안해 드리겠습니다.

---

## 1단계: 현재 재정 상태 파악하기 (진단)

저축을 늘리기 전에, **돈이 어디로 가고 있는지** 정확히 아는 것이 가장 중요합니다.

1. **수입과 지출 기록:** 지난 한 달간의 모든 수입(월급, 부수입 등)과 지출 내역(식비, 교통비, 구독료, 여가 활동 등)을 꼼꼼하게 기록해 보세요. 가계부 앱이나 엑셀을 활용하면 편리합니다.
2. **'돈의 흐름' 분석:** 지출 내역을 분석하여 **고정 지출** (월세, 보험료, 통신비 등)과 **변동 지출** (외식비, 쇼핑, 취미 등)을 구분합니다.
3. **불필요한 지출 식별:** 기록을 통해 '이 돈은 정말 필요한가?' 싶은 항목(예: 사용하지 않는 구독 서비스, 습관적인 충동구매)을 찾아냅니다.

## 2단계: 예산 설정 및 목표 구체화 (계획)

막연하게 저축하겠다는 것보다 **구체적인 목표**가 있어야 동기 부여가 됩니다.

1. **'선 저축, 후 소비' 원칙 적용:** 월급이 들어오자마자 저축할 금액을 먼저 떼어놓고, 남은 돈으로 생활하는 습관을 들여야 합니다.
2. **50/30/20 법칙 활용 (가이드라인):**
    * **50%:** 필수 지출 (주거비, 식비, 공과금 등)
    * **30%:** 원하는 지출 (취미, 외식, 쇼핑 등)
    * **20% 이상:** 저축 및 투자 (이 부분을 최우선으로 늘려야 합니다.)
3. **단기/장기 목표 설정:**
    * **단기 목표 (3~6개월):** 비상금 마련 (예: 3개월치 생활비)
    * **중기 목표 (1~3년):** 특정 목적 자금 (예: 여행 자금, 자동차 구매 자금)
    * **장기 목표 (5년 이상):** 은퇴 자금, 주택 마

In [20]:

result = trim_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?")
        ]
    },
    trim_config,
)

print_last_answer(result)

집을 구매하는 것은 인생에서 매우 중요한 결정이며, 이를 위해서는 체계적이고 철저한 재정 계획이 필수적입니다. 5년이라는 기간은 목표 달성을 위해 충분하지만, 구체적인 계획이 없다면 막막할 수 있습니다.

제가 금융 상담사로서 고객님의 성공적인 주택 구매를 돕기 위해 **단계별 재정 계획 로드맵**을 제시해 드리겠습니다. 이 계획은 크게 **준비 단계, 실행 단계, 유지 및 관리 단계**로 나눌 수 있습니다.

---

## 🏡 5년 내 주택 구매를 위한 재정 계획 로드맵

### 1단계: 현재 재정 상태 파악 및 목표 설정 (Foundation)

가장 먼저 '현재 위치'를 정확히 알아야 어디로 가야 할지 알 수 있습니다.

**1. 순자산 및 현금 흐름 분석:**
* **순자산 계산:** 현재 보유한 모든 자산(예금, 투자금, 기타 자산)에서 부채(대출 등)를 뺀 금액을 파악합니다.
* **월별 현금 흐름 분석 (가장 중요):** 수입과 지출을 꼼꼼히 기록하여 매달 얼마를 저축하고 투자할 수 있는지 정확히 알아야 합니다. 불필요한 지출을 줄일 여지를 찾습니다.

**2. 목표 주택 가격 및 예산 설정:**
* **희망 지역 및 가격대 설정:** 어느 지역에서, 대략 얼마짜리 집을 구매하고 싶은지 현실적인 범위를 정합니다. (지역별 시세 조사 필수)
* **총 필요 자금 계산:**
    * **계약금/초기 비용:** 주택 가격의 5~10% (취득세, 중개 수수료 등 포함)
    * **대출 가능 금액 예측:** 현재 소득과 신용도를 바탕으로 은행에서 받을 수 있는 최대 대출 한도를 미리 시뮬레이션 해봅니다.

**3. 비상 자금 확보:**
* 주택 구매 외에 예상치 못한 지출(가전제품 고장, 갑작스러운 실직 등)에 대비하여 **최소 6개월치 생활비**를 별도의 안전 자산으로 확보해야 합니다.

---

### 2단계: 자금 마련 전략 수립 (Fueling the Journey)

목표 금액을 달성하기 위해 '어떻게 돈을 모을 것인가'에 대한 구체적인 전략

In [21]:
result = trim_graph.invoke(
    {
        "messages": [
            HumanMessage(content="내가 방금 뭐라고 했나요?")
        ]
    },
    trim_config,
)

print_last_answer(result)

방금 **"내가 방금 뭐라고 했나요?"** 라고 질문하셨습니다. 😊

혹시 이전에 저에게 하셨던 말씀이나, 제가 드린 답변 중에서 다시 확인하고 싶으신 내용이 있으신가요? 구체적으로 어떤 내용을 찾고 계신지 말씀해주시면 정확하게 다시 알려드리겠습니다!


In [22]:
print_graph_state(trim_graph, trim_config)


===== 현재 summary =====


===== 현재 messages =====
human: 저축을 늘리기 위해 무엇을 할 수 있나요?
--------------------------------------------------
ai: 저축을 늘리는 것은 재정적 목표를 달성하는 데 매우 중요합니다. 단순히 '돈을 아껴야지'라고 생각하기보다는, **체계적인 계획과 습관의 변화**가 필요합니다.

제가 금융 상담사로서, 저축액을 효과적으로 늘릴 수 있는 구체적이고 실질적인 방법들을 단계별로 제안해 드리겠습니다.

---

## 1단계: 현재 재정 상태 파악하기 (진단)

저축을 늘리기 전에, **돈이 어디로 가고 있는지** 정확히 아는 것이 가장 중요합니다.

1. **수입과 지출 기록:** 지난 한 달간의 모든 수입(월급, 부수입 등)과 지출 내역(식비, 교통비, 구독료, 여가 활동 등)을 꼼꼼하게 기록해 보세요. 가계부 앱이나 엑셀을 활용하면 편리합니다.
2. **'돈의 흐름' 분석:** 지출 내역을 분석하여 **고정 지출** (월세, 보험료, 통신비 등)과 **변동 지출** (외식비, 쇼핑, 취미 등)을 구분합니다.
3. **불필요한 지출 식별:** 기록을 통해 '이 돈은 정말 필요한가?' 싶은 항목(예: 사용하지 않는 구독 서비스, 습관적인 충동구매)을 찾아냅니다.

## 2단계: 예산 설정 및 목표 구체화 (계획)

막연하게 저축하겠다는 것보다 **구체적인 목표**가 있어야 동기 부여가 됩니다.

1. **'선 저축, 후 소비' 원칙 적용:** 월급이 들어오자마자 저축할 금액을 먼저 떼어놓고, 남은 돈으로 생활하는 습관을 들여야 합니다.
2. **50/30/20 법칙 활용 (가이드라인):**
    * **50%:** 필수 지출 (주거비, 식비, 공과금 등)
    * **30%:** 원하는 지출 (취미, 외식, 쇼핑 등)
    * **20% 이상:** 저축 및 투자 (이 부분을 최우선으로 늘려야 합니다.)
3. **단기/장기 목표 설정:**
 

In [23]:
# ============================================================
# 실행 예시 3. summarization 적용
# ============================================================

print("\n\n==============================")
print("3. summarization 적용")
print("==============================")

summary_config = {
    "configurable": {
        "thread_id": "summary_finance_session"
    }
}

result = summary_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저축을 늘리기 위해 무엇을 할 수 있나요?")
        ]
    },
    summary_config,
)
print_last_answer(result)



3. summarization 적용
저축을 늘리는 것은 재정적 목표를 달성하는 데 매우 중요합니다. 단순히 '돈을 아껴야지'라고 생각하기보다는, **체계적인 계획과 습관의 변화**가 필요합니다.

제가 금융 상담사로서 저축을 늘리기 위해 실천할 수 있는 구체적이고 효과적인 방법들을 단계별로 안내해 드리겠습니다.

---

## 1단계: 현재 재정 상태 파악하기 (진단)

저축을 늘리기 전에, **돈이 어디로 가고 있는지** 정확히 아는 것이 가장 중요합니다.

1. **수입과 지출 기록:** 최소 한 달 동안 모든 수입(월급, 부수입 등)과 모든 지출(고정비, 변동비 포함)을 꼼꼼하게 기록해 보세요. 가계부 앱이나 엑셀을 활용하면 편리합니다.
2. **지출 분석 및 분류:** 기록한 내역을 항목별로 분류합니다. (예: 주거비, 식비, 교통비, 취미/여가, 구독 서비스 등)
3. **'돈 새는 구멍' 찾기:** 지출 내역을 보면서 불필요하거나 과도하게 지출되는 항목(예: 습관적인 커피값, 사용하지 않는 구독 서비스)을 찾아냅니다.

## 2단계: 예산 설정 및 목표 설정하기 (계획)

분석이 끝났다면, 이제 저축을 위한 구체적인 계획을 세워야 합니다.

1. **'선 저축, 후 소비' 원칙 적용:** 월급이 들어오자마자 저축할 금액을 먼저 따로 떼어 놓으세요. 남은 돈으로 생활하는 습관을 들여야 합니다.
2. **구체적인 저축 목표 설정 (SMART 원칙):** 막연히 '저축을 늘리겠다'가 아니라, "6개월 안에 비상금 300만 원 모으기", "1년 안에 여행 자금 500만 원 마련하기"와 같이 구체적이고 측정 가능한 목표를 세우세요.
3. **예산 배분:** 설정한 목표에 맞춰 수입을 다음과 같이 배분합니다.
    * **필수 지출 (고정비):** 월세, 공과금, 보험료 등
    * **저축/투자 (목표 금액):** 매월 얼마를 저축할지 결정
    * **변동 지출 (생활비):** 식비, 여가비 등

## 3단계: 지출 통제 및 절약 실행하기 (실행)

In [24]:
result = summary_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?")
        ]
    },
    summary_config,
)
print_last_answer(result)

5년이라는 구체적인 목표를 가지고 집을 구매하려는 계획은 매우 훌륭하고 현실적입니다. 주택 구매는 큰 자금이 필요하기 때문에, **단순히 저축하는 것을 넘어 '목표 달성을 위한 전략적인 재정 설계'**가 필수적입니다.

금융 상담사로서, 5년 내 주택 구매를 위해 세워야 할 핵심 재정 계획을 **3단계 로드맵**으로 나누어 상세하게 안내해 드리겠습니다.

---

## 🏠 5년 내 주택 구매를 위한 3단계 재정 로드맵

### 1단계: 현재 상황 진단 및 목표 금액 설정 (Foundation)

가장 먼저, '무엇을', '얼마나' 모아야 하는지 정확히 알아야 합니다.

#### 1. 목표 주택 가격 및 필요 자금 계산
* **희망 주택 가격 설정:** 원하는 지역과 규모를 고려하여 현실적인 최대 예산을 정합니다. (매매가 + 취득세, 중개 수수료 등 부대 비용 포함)
* **필요 초기 자금 파악:**
    * **계약금/초기 비용:** 주택 가격의 10~20% 정도를 준비해야 합니다.
    * **대출 가능 금액 확인:** 현재 소득과 신용도를 바탕으로 은행에서 받을 수 있는 최대 대출 한도를 미리 시뮬레이션 해보세요. (DSR 규제 등을 고려)
    * **비상 자금 확보:** 주택 구매 후에도 발생할 수 있는 예상치 못한 지출(수리비, 이사 비용 등)을 위한 별도의 비상금을 마련해야 합니다.

#### 2. 현재 재정 상태 분석 (현실 점검)
* **순자산 계산:** 현재 보유한 자산(예금, 투자금, 기타 자산)에서 부채(신용대출, 학자금 대출 등)를 뺀 순자산을 파악합니다.
* **월별 현금 흐름 분석:** 현재 수입에서 고정 지출과 변동 지출을 제외하고, **매월 저축 및 투자에 투입할 수 있는 '잉여 자금'**이 얼마인지 정확히 계산해야 합니다.

### 2단계: 공격적인 저축 및 부채 관리 (Acceleration)

목표 달성을 위해 현재의 재정 상태를 최대한 끌어올리는 단계입니다.

#### 1. 지출 극단적 통제 (가장 중요)
* **'집

In [25]:
result = summary_graph.invoke(
    {
        "messages": [
            HumanMessage(content="월급의 일부를 자동이체로 저축하는 방식도 괜찮을까요?")
        ]
    },
    summary_config,
)
print_last_answer(result)

네, **월급의 일부를 자동이체로 저축하는 방식은 재정 계획에서 가장 강력하고 효과적인 방법 중 하나**입니다. 금융 전문가들이 가장 추천하는 '선(先) 저축, 후(後) 소비' 원칙을 실현하는 핵심 메커니즘이기 때문입니다.

하지만 이 방식이 **최적의 방법**이 되기 위해서는 몇 가지 중요한 조건과 고려사항이 있습니다.

---

## ✅ 자동이체 저축 방식의 장점 (Why it works)

1. **강제성 확보 (Discipline):** 돈을 눈으로 보고 소비하기 전에 저축 계좌로 빠져나가게 만듭니다. 이는 충동적인 지출을 막고 계획된 저축액을 반드시 확보하게 해주는 가장 강력한 심리적 장치입니다.
2. **자동화 및 편리성:** 매번 수동으로 이체할 필요가 없어 관리가 매우 쉽습니다. '잊어버리는' 것을 방지해 줍니다.
3. **일관성 유지:** 매달 일정한 금액이 빠져나가기 때문에 저축 습관을 꾸준히 유지하는 데 최적입니다.
4. **복리의 효과 시작:** 돈이 바로 투자나 저축 계좌로 이동하므로, 복리 효과를 최대한 빨리 누리기 시작할 수 있습니다.

## ⚠️ 성공적인 자동이체 저축을 위한 필수 조건 (How to make it successful)

단순히 '자동이체'만 설정하는 것만으로는 부족합니다. 다음 세 가지 요소를 반드시 점검해야 합니다.

### 1. 목표 금액의 현실성 (The Right Amount)
* **현실적인 비율:** 자동이체로 설정할 금액은 현재의 수입 수준에서 **지속 가능**해야 합니다. 너무 공격적으로 설정하면 생활이 어려워져 결국 포기하게 됩니다.
* **최소한의 생계비 확보:** 저축액을 정할 때, 나머지 금액으로도 최소한의 생활(식비, 공과금 등)이 충분히 해결되는지 반드시 확인해야 합니다.

### 2. 계좌 분리 (Separation of Funds)
* **'보이지 않는 돈' 만들기:** 자동이체로 저축하는 돈은 **일상적인 소비 계좌와 완전히 분리된 별도의 저축/투자 계좌**에 넣어두어야 합니다. 만약

In [26]:
result = summary_graph.invoke(
    {
        "messages": [
            HumanMessage(content="대출을 활용해서 더 빨리 집을 사는 것은 위험할까요?")
        ]
    },
    summary_config,
)
print_last_answer(result)

# 이 시점쯤 messages가 6개를 넘기 때문에 summarize 노드가 실행될 수 있습니다.

대출을 활용하여 주택 구매 시기를 앞당기는 것은 **'기회'와 '위험'이 공존하는 매우 신중하게 접근해야 하는 전략**입니다. 단순히 대출을 받는 행위 자체가 위험한 것은 아니지만, **어떤 종류의 대출을, 어떤 조건으로 받느냐**에 따라 미래 재정에 치명적인 영향을 줄 수 있습니다.

금융 상담사로서, 이 전략의 장점과 반드시 고려해야 할 위험 요소를 명확하게 분석해 드리겠습니다.

---

## ⚖️ 대출 활용 전략: 장점 vs. 위험 요소 분석

### 👍 대출 활용의 잠재적 장점 (The Upside)

1. **시간 단축:** 가장 큰 장점입니다. 저축만으로 5년이 걸릴 목표를, 대출을 통해 2~3년으로 단축하여 원하는 시점에 주택을 소유할 수 있습니다.
2. **자산 가치 상승 기회 포착:** 부동산 시장이 상승기에 있다면, 대출을 활용해 자산을 확보함으로써 자산 증식의 이익을 극대화할 수 있습니다.
3. **기회비용 최소화:** 만약 현재 시점에서 집값이 급등할 것으로 예상된다면, 기다리는 것보다 선제적으로 움직이는 것이 유리할 수 있습니다.

### 🚨 대출 활용의 치명적인 위험 요소 (The Downside)

#### 1. 이자 비용의 누적 (Cost of Debt)
* **총 이자 부담 증가:** 대출 기간 동안 발생하는 이자는 원금에 더해져 최종적으로 내가 부담해야 할 총비용을 크게 늘립니다. 특히 금리가 높은 시기에 대출을 받으면 그 부담이 가중됩니다.
* **상환 압박:** 매달 고정적인 이자와 원금을 갚아야 하므로, 예상치 못한 소득 감소나 경제 상황 악화 시 상환에 큰 압박을 받을 수 있습니다.

#### 2. 금리 변동 위험 (Interest Rate Risk)
* **변동금리 리스크:** 변동금리를 선택할 경우, 시장 금리가 상승하면 매달 내야 하는 이자액이 급격히 늘어날 수 있으며, 이는 예산 계획을 완전히 무너뜨릴 수 있습니다.

#### 3. 재정적 유연성 상실 (Loss of Flexibility)
* **재정적 경직성:

In [27]:
result = summary_graph.invoke(
    {
        "messages": [
            HumanMessage(content="지금까지 저에게 어떤 재정적 조언을 해주셨나요?")
        ]
    },
    summary_config,
)
print_last_answer(result)

제가 지금까지 사용자님께 드린 재정적 조언은 **'목표 설정 → 실행 계획 수립 → 위험 관리'**라는 3단계의 체계적인 접근 방식을 중심으로 이루어졌습니다.

사용자님의 궁극적인 목표인 **"5년 내 주택 마련"**을 달성하기 위해, 저는 다음과 같은 핵심적인 조언들을 단계별로 제공해 드렸습니다.

---

### 1. 🎯 초기 단계: 명확한 목표 설정 및 기반 다지기 (Foundation Building)

가장 먼저, 막연한 희망이 아닌 **구체적인 수치**를 설정하도록 조언했습니다.

* **목표 금액 구체화:** "어떤 가격대의 집을 살 것인지", 그리고 그 집을 사기 위해 **총 얼마의 자금이 필요한지**를 명확히 계산해야 한다고 강조했습니다.
* **수입 증대 모색:** 저축만으로는 목표 달성이 어려울 수 있으므로, 저축 속도를 높이기 위해 **추가적인 수입 파이프라인을 찾는 것**이 중요하다고 조언했습니다.
* **투자보다 안전 자산 중심 운용:** 주택 구매라는 명확한 단기/중기 목표가 있기 때문에, 높은 수익률을 쫓는 위험한 투자보다는 **예금이나 적금 등 안전 자산 위주로 자금을 운용**해야 한다고 안내했습니다.

### 2. 🛠️ 실행 단계: 저축 습관의 자동화 (Execution & Habit Formation)

계획을 세웠다면, 그것을 꾸준히 실천할 수 있는 시스템을 구축하는 것이 중요합니다.

* **자동이체 시스템 도입 권장:** 월급에서 일정 금액을 **자동으로 저축 계좌로 이체**하는 방식을 강력하게 추천했습니다. 이는 의지력에만 의존하는 것을 막고, 저축을 '강제적인 습관'으로 만드는 가장 효과적인 방법이라고 설명드렸습니다.
* **자금 분리 원칙 강조:** 저축할 돈과 일상 소비할 돈의 계좌를 **분리**하여, 실수로 자금을 유용하거나 소비하는 것을 방지해야 한다고 조언했습니다.
* **비상금 확보 선행:** 갑작스러운 지출에 대비하여 **충분한 비상금을 먼저 확보**한 후에 본격적인 주택 마련 저축을 시작해야 한다는 안전

In [28]:
print_graph_state(summary_graph, summary_config)


===== 현재 summary =====
## 📝 대화 요약

이 대화는 **사용자가 주택 구매를 위한 저축 전략**에 대해 문의하고, 이에 대한 구체적인 실행 방안과 위험 관리에 대해 심층적으로 논의하는 과정이었습니다.

---

### 🎯 사용자의 핵심 목표

* **궁극적 목표:** 저축을 늘려 궁극적으로 집을 구매하는 것 (구체적으로는 5년 내 주택 마련을 염두에 두고 있음).

### 💬 주요 질문 및 논의 주제

1. **저축 증대 방법 문의:** 저축을 늘리기 위해 구체적으로 무엇을 해야 하는지에 대한 조언을 요청했습니다.
2. **자동이체 방식 검토:** 월급의 일부를 자동이체로 저축하는 방식이 적절한지 확인했습니다.
3. **대출 활용 위험성 질문:** 대출을 활용하여 집 구매 시기를 앞당기는 것이 위험한지에 대해 문의했습니다.

### 💡 AI가 제공한 핵심 재정 조언 요약

#### 1. 저축 전략 (초기 단계)
* **구체적인 계획 수립:** 목표 주택 가격과 필요한 총액을 구체적으로 설정해야 합니다.
* **수입 증대 모색:** 저축 속도를 높이기 위해 추가적인 수입 파이프라인을 찾는 것이 중요합니다.
* **안전 자산 중심 운용:** 장기 목표이므로 투자보다는 예금/적금 등 안전 자산 위주로 운용해야 합니다.

#### 2. 자동이체 방식에 대한 조언 (실행 단계)
* **장점 강조:** 강제성을 확보하고 습관을 유지하는 데 매우 효과적인 방법임을 설명했습니다.
* **성공 조건 제시:**
    * 목표 금액의 현실성 확인.
    * 저축 계좌와 소비 계좌를 **분리**하여 자금 유용 방지.
    * 비상금 확보 후 실행하는 것이 안전함.

#### 3. 대출 활용에 대한 조언 (고급 전략 단계)
* **균형 잡힌 접근:** 대출은 '기회'가 될 수 있지만, **'위험'도 매우 크다**는 점을 강조했습니다.
* **위험 요소 경고:**
    * **이자 비용 누적:** 총비용 증가 및 상환 압박.
    * **금리 변동 위험

In [29]:
# ============================================================
# 실행 예시 4. thread_id 분리 확인
# ============================================================
#
# 같은 graph라도 thread_id가 다르면 서로 다른 대화 이력을 가집니다.
# 기존 15-message-trimming-and-summarization의 session_id="unused"와 달리,
# LangGraph의 thread_id는 실제로 사용됩니다.

print("\n\n==============================")
print("4. thread_id 분리 확인")
print("==============================")

user_a_config = {
    "configurable": {
        "thread_id": "user_a"
    }
}

user_b_config = {
    "configurable": {
        "thread_id": "user_b"
    }
}



4. thread_id 분리 확인


In [30]:
basic_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저는 매달 50만 원씩 저축하고 싶습니다.")
        ]
    },
    user_a_config,
)

{'messages': [HumanMessage(content='저는 매달 50만 원씩 저축하고 싶습니다.', additional_kwargs={}, response_metadata={}, id='bfc64c56-f90e-41ba-8100-82438f0bbe7c'),
  AIMessage(content="매달 50만 원씩 저축하시려는 계획, 정말 훌륭합니다! 꾸준한 저축은 재정 목표 달성에 매우 중요한 첫걸음입니다.\n\n하지만 제가 더 구체적이고 실질적인 조언을 드리기 위해서는 몇 가지 추가 정보가 필요합니다. 현재 고객님의 **재정 상황**과 **목표**에 따라 가장 적합한 저축 방법이 달라지기 때문입니다.\n\n혹시 아래 질문들에 대해 조금 더 자세히 말씀해 주실 수 있을까요?\n\n### 1. 저축의 '목표'는 무엇인가요?\n* **단기 목표 (1~3년 이내):** 비상금 마련, 여행 자금, 특정 물건 구매 등\n* **중기 목표 (3~7년):** 주택 계약금, 자동차 구매, 목돈 마련 등\n* **장기 목표 (7년 이상):** 은퇴 자금, 자녀 교육 자금 등\n\n### 2. 현재 '재정 상황'은 어떠신가요?\n* **월 수입:** 매달 고정적으로 들어오는 수입은 어느 정도인가요?\n* **현재 지출:** 매달 고정적으로 나가는 필수 지출(월세/대출 이자, 공과금, 식비 등)은 대략 어느 정도인가요?\n* **기존 저축/투자 현황:** 현재 다른 곳에 돈을 모으고 계신 것이 있나요?\n\n### 3. '저축의 성격'은 무엇인가요?\n* **안정성 중시:** 원금 손실 없이 안전하게 보관하고 싶으신가요? (예: 예금, 적금)\n* **수익성 추구:** 약간의 이자 수익이라도 얻고 싶으신가요? (예: 펀드, ETF 등)\n\n---\n\n**💡 일단, 목표를 정하기 전까지 고려해 볼 수 있는 일반적인 조언 세 가지를 먼저 드립니다.**\n\n### 1. '선 저축, 후 소비' 습관 확립\n매달 50만 원을 **가장 먼저** 저축 계좌로 이체하는 자동이체를

In [31]:
basic_graph.invoke(
    {
        "messages": [
            HumanMessage(content="저는 투자보다 예금을 선호합니다.")
        ]
    },
    user_b_config,
)

{'messages': [HumanMessage(content='저는 투자보다 예금을 선호합니다.', additional_kwargs={}, response_metadata={}, id='0058ca86-cdc7-4d47-a0bc-a97b544feb7b'),
  AIMessage(content='투자보다는 예금을 선호하시는군요. 이는 **안정성과 원금 보존**을 매우 중요하게 생각하신다는 의미로 이해됩니다. 많은 분들이 재정적 안정성을 위해 예금 상품을 선호하시는데, 어떤 점에 중점을 두고 계신지 조금 더 자세히 말씀해주시면, 고객님께 가장 적합한 예금 상품이나 재정 계획에 대해 구체적인 도움을 드릴 수 있습니다.\n\n예를 들어, 다음과 같은 부분에 대해 궁금하신가요?\n\n1. **어떤 종류의 예금을 선호하시나요?** (예: 일반 정기예금, 특판 예금, 고금리 적금, 파킹 통장 등)\n2. **투자 대비 수익률(이자율)이 어느 정도 중요하신가요?** (안정성을 최우선으로 하더라도, 현재 시장 상황에서 어느 정도의 이자 수익을 기대하시는지 궁금합니다.)\n3. **예금 기간은 얼마나 생각하고 계신가요?** (단기, 중기, 장기)\n4. **원금 손실 위험에 대해 얼마나 민감하신가요?**\n\n이러한 정보를 알려주시면, 현재 금리 상황을 고려하여 **안정성, 유동성, 그리고 수익성**의 균형을 맞출 수 있는 최적의 예금 전략을 함께 고민해 드리겠습니다. 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 295, 'prompt_tokens': 41, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': Non

In [32]:
result_a = basic_graph.invoke(
    {
        "messages": [
            HumanMessage(content="내가 방금 뭐라고 했나요?")
        ]
    },
    user_a_config,
)

In [33]:
result_b = basic_graph.invoke(
    {
        "messages": [
            HumanMessage(content="내가 방금 뭐라고 했나요?")
        ]
    },
    user_b_config,
)

In [34]:
print("\n[user_a 응답]")
print_last_answer(result_a)


[user_a 응답]
고객님께서는 **"저는 매달 50만 원씩 저축하고 싶습니다."** 라고 말씀하셨습니다.

제가 그 말씀에 대해 더 구체적인 조언을 드리기 위해, 현재 고객님의 재정 상황과 저축 목표에 대해 몇 가지 질문을 드렸습니다.


In [35]:
print("\n[user_b 응답]")
print_last_answer(result_b)


[user_b 응답]
고객님께서는 **"저는 투자보다 예금을 선호합니다."** 라고 말씀하셨습니다.

제가 그 말씀을 듣고, 고객님의 선호도(안정성 중시)를 바탕으로 어떤 종류의 예금 상품이나 재정 계획에 대해 더 자세히 알고 싶다는 질문을 드렸습니다.
